<a href="https://colab.research.google.com/github/mukailaalhshituabr-cyber/lab-4-llm-decision-support/blob/main/Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Part 0: Repository and API-key setup
# API-key setup cell
import os

try:
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.environ["GROQ_API_KEY"]

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


Section 1 — Talking to an LLM Programmatically

In [14]:
# Part 1.1 — Your first API call
# TODO: Write a helper function you will reuse for the WHOLE lab:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

question = "What is microfinance, in one sentence?"
print(ask_llm(question))

# Call the API directly (not through the helper) so we can inspect token usage
# TODO: Call it once with a simple question and print the answer.
raw_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
)
# TODO: Print response.usage as well — how many tokens did your call consume?
print(raw_response.usage)

Microfinance refers to the provision of small loans, savings, and other financial services to low-income individuals or groups who lack access to traditional banking services, often in developing countries.
CompletionUsage(completion_tokens=47, prompt_tokens=44, total_tokens=91, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.061962997, prompt_time=0.001959877, completion_time=0.151044929, total_time=0.153004806)


system vs user: system sets persistent behaviour/rules for the whole conversation (e.g. "be factual and neutral"). user carries the specific request for this turn (the letter text, or a question).
What is a token: roughly a chunk of text, often about 3/4 of a word. Providers bill per token because token count drives the actual compute cost, every token is processed by the model, so per-token billing ties price to the work done rather than a flat fee per request.

In [15]:
# Part 1.2 — Temperature: the randomness dial

# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = [ask_llm(question, temperature=0.0) for _ in range(5)]
high_temp_answers = [ask_llm(question, temperature=1.2) for _ in range(5)]

# TODO: Print all 10 answers, grouped by temperature.
print("temperature = 0.0 ")
for i, a in enumerate(low_temp_answers, 1):
    print(f"{i}. {a}\n")

print("temperature = 1.2 ")
for i, a in enumerate(high_temp_answers, 1):
    print(f"{i}. {a}\n")

temperature = 0.0 
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "save" or "keep", so this name is simple and straightforward.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Adanfo Account**: "Adanfo" is a Ghanaian word for "friends" or "partners", which could em

At 0.0 the 5 answers are identical or nearly identical, the model deterministically picks its highest-probability words every time. At 1.2 the 5 answers vary noticeably in wording and even in the name suggested. For the loan decision-support system, low temperature (0) is the right choice for summarization, extraction, and briefs: the officer needs the same letter to produce the same facts every run, creativity here only adds inconsistency and hallucination risk, not value.

Section 2 — The Dataset: Loan Application Letters

In [16]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.



Section 3 — Prompt Engineering for the Decision Support System

# Part 3.1 — Component 1: Summarization
SUMMARY_PROMPT_V1 = "Summarize this:"

def summarize_v1(letter_text):
    return ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}")

print("L002 (V1)")
print(summarize_v1(LETTERS["L002"]))
print("\nL006 (V1)")
print(summarize_v1(LETTERS["L006"]))

SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan application letters factually and neutrally. "
    "Do not invent, assume, or embellish any detail that is not explicitly "
    "stated in the letter. Write exactly 3-4 sentences."
)
SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

def summarize_v2(letter_text):
    user_prompt = SUMMARY_PROMPT_V2.format(letter_text=letter_text)
    return ask_llm(user_prompt, system_prompt=SUMMARY_SYSTEM_V2, temperature=0)

print("L002 (V2)")
print(summarize_v2(LETTERS["L002"]))
print("\nL006 (V2)")
print(summarize_v2(LETTERS["L006"]))

for letter_id in ["L002", "L006"]:
    print(f"{letter_id}")
    print("V1:", summarize_v1(LETTERS[letter_id]))
    print("V2:", summarize_v2(LETTERS[letter_id]))
    print()

V1 typically rambles, may drift into opinion, or misses key facts (amount, purpose) buried in the letter, with no consistent length. V2 fixes this with the explicit role + length/factuality constraint, producing a consistent, scannable brief every time.
The officer acts on this summary, so an invented detail (e.g. a made-up profit figure) could lead to a wrong lending decision. This failure mode is called hallucination in the LLM literature.

In [17]:
# Part 3.2 — Component 2: Structured extraction (JSON)

import json

EXTRACT_SYSTEM = """You are a data extraction assistant for a microfinance loan officer.
Extract information from a loan application letter and return ONLY a JSON object
with EXACTLY these keys, no others:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

If a field is not stated in the letter, use null. Do not guess.
Return ONLY the JSON object -- no extra text, no markdown code fences."""

EXTRACT_PROMPT = """Here is a worked example.

Letter:
Dear Sir, I am Ama Nyarko, a hairdresser in Cape Coast. I need GHS 5,000 to buy new
dryers. My salon currently makes about GHS 600 profit a month. I have no collateral
to offer. I can repay GHS 300 monthly for 18 months.

JSON:
{{
  "applicant_name": "Ama Nyarko",
  "amount_ghs": 5000,
  "purpose": "buy new dryers",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": false,
  "repayment_months": 18
}}

Now extract the same fields from this new letter. Return ONLY the JSON object.

Letter:
{letter_text}

JSON:"""

def extract_fields(letter_text, temperature=0):
    user_prompt = EXTRACT_PROMPT.format(letter_text=letter_text)
    raw = ask_llm(user_prompt, system_prompt=EXTRACT_SYSTEM, temperature=temperature)

    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"Could not parse JSON. Raw output was:\n{raw}")
        return None

import pandas as pd

rows = []
for letter_id, text in LETTERS.items():
    fields = extract_fields(text)
    if fields is not None:
        fields["letter_id"] = letter_id
        rows.append(fields)

extracted_df = pd.DataFrame(rows).set_index("letter_id")
extracted_df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


If the few-shot example were one of the six target letters, the model could pattern-match an answer it already effectively sawdd, and it would contaminate the Section 4 evaluation, since you'd be grading the model on a letter it was handed the answer key for.
Without that instruction, the model tends to invent plausible-but-false values for missing fields (e.g. guessing a profit figure for L002, which never states one) instead of honestly reporting the gap.
temperature=0 suits extraction because there's exactly one correct set of facts to pull from fixed textdd, we want the most likely, reproducible answer, not variety. Creative tasks benefit from randomness; a single verifiable fact does not.

In [18]:
# Part 3.3 — Component 3: The decision-support brief
BRIEF_SYSTEM = """You are a decision-support assistant for a microfinance loan officer
in Ghana. You NEVER approve or reject a loan yourself; that decision is always made
by a human loan officer. Given a letter and its extracted data, respond with exactly
four sections:
1. Strengths (bullet points, grounded only in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents",
   "flag for senior review"); NEVER "approve" or "reject"."""

BRIEF_PROMPT = """Letter:
{letter_text}

Extracted data:
{extracted_json}

Write the four-section brief described in your instructions."""

def generate_brief(letter_text, extracted_fields):
    user_prompt = BRIEF_PROMPT.format(
        letter_text=letter_text,
        extracted_json=json.dumps(extracted_fields, indent=2),
    )
    return ask_llm(user_prompt, system_prompt=BRIEF_SYSTEM, temperature=0)

briefs = {}
for letter_id, text in LETTERS.items():
    fields = extracted_df.loc[letter_id].to_dict() if letter_id in extracted_df.index else {}
    briefs[letter_id] = generate_brief(text, fields)

for letter_id in ["L001", "L002", "L006"]:
    print(f"BRIEF: {letter_id}")
    print(briefs[letter_id])
    print()

BRIEF: L001
## Step 1: Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable and established business.
* She has a consistent profit of GHS 900 per month, demonstrating a viable income stream.
* Akosua has saved GHS 2,500 through the susu scheme over two years without missing a contribution, showing discipline and ability to manage savings.
* She has a guarantor, her sister, who is a teacher, providing an added layer of security for the loan.

## Step 2: Risks / red flags
* The loan amount of GHS 8,000 is significant compared to her monthly profit, which might pose a risk if her business does not expand as planned.
* Expanding into frozen foods with a deep freezer requires additional skills and knowledge, and there's a risk that Akosua may not successfully adapt to this new venture.
* The repayment plan of GHS 450 per month over 20 months is relatively high compared to her current profit, leaving little room for er

L003's brief should surface strong strengths (registered business, 18 months of records, existing collateral, realistic repayment plan) with few red flags; L006's should be thin on strengths and heavy on red flags (no existing business, no collateral, three unrelated ventures at once, vague repayment). If your actual output matches this pattern, the system is surfacing the right signals. Practical: the model can miss context or hallucinate, so a wrong automated decision risks real financial harm and liability. Ethical: decisions that affect people's access to credit should stay accountable to a human who can be questioned and exercise judgement a model can't.

Section 4 — Evaluation: Quality, Reliability, Appropriateness

In [19]:
# Part 4.1 — Extraction accuracy against gold labels
FIELDS = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

def values_match(field, predicted, gold):
    if predicted is None and gold is None:
        return True
    if predicted is None or gold is None:
        return False
    if field == "applicant_name":
        return str(predicted).strip().lower() == str(gold).strip().lower()
    if field == "purpose":
        p, g = str(predicted).strip().lower(), str(gold).strip().lower()
        return g in p or p in g
    return predicted == gold

results = {}
for letter_id, gold_fields in GOLD.items():
    predicted_fields = extracted_df.loc[letter_id].to_dict()
    results[letter_id] = {
        field: values_match(field, predicted_fields.get(field), gold_fields[field])
        for field in FIELDS
    }

accuracy_table = pd.DataFrame(results)
accuracy_table["accuracy"] = accuracy_table.mean(axis=1)
accuracy_table


,L001,L003,L006,accuracy
applicant_name,True,True,True,1.000000
amount_ghs,True,True,True,1.000000
purpose,False,True,False,0.333333
monthly_profit_ghs,True,True,False,0.666667
has_collateral_or_guarantor,True,True,True,1.000000
repayment_months,True,True,True,1.000000


In [20]:
# Part 4.2 — Reliability: is the system consistent?
def json_signature(result):
    if result is None:
        return None
    return json.dumps(result, sort_keys=True)

low_temp_runs = [extract_fields(LETTERS["L004"], temperature=0.0) for _ in range(5)]
high_temp_runs = [extract_fields(LETTERS["L004"], temperature=1.0) for _ in range(5)]

def summarize_reliability(runs, label):
    valid = [r for r in runs if r is not None]
    unique_signatures = set(json_signature(r) for r in valid)
    print(f"--- temperature = {label} ---")
    print(f"Valid JSON: {len(valid)}/5")
    print(f"Unique outputs among valid: {len(unique_signatures)}/{len(valid)}")

summarize_reliability(low_temp_runs, "0.0")
summarize_reliability(high_temp_runs, "1.0")

--- temperature = 0.0 ---
Valid JSON: 5/5
Unique outputs among valid: 1/5
--- temperature = 1.0 ---
Valid JSON: 5/5
Unique outputs among valid: 2/5


In [21]:
# Part 4.3 — Hallucination probing
test1_question = "What is the applicant's credit score?"
test1_prompt = f"{LETTERS['L001']}\n\nQuestion: {test1_question}"
test1_answer = ask_llm(
    test1_prompt,
    system_prompt="Answer only using information stated in the letter above. "
                  "If the information is not present, say so explicitly.",
    temperature=0,
)
print("TEST 1 ANSWER:\n", test1_answer)
# Record verbatim below and label PASS (says "not stated") or FAIL (invents a score).

# Test 2 feed the extractor something completely irrelevant
irrelevant_text = ("Weather report: Accra will see scattered showers this afternoon "
                    "with a high of 29C and light winds from the southwest.")
test2_result = extract_fields(irrelevant_text)
print("\nTEST 2 RESULT:\n", test2_result)
# Record verbatim below and label PASS (all fields null / refuses) or FAIL (invents an applicant).

TEST 1 ANSWER:
 The information about the applicant's credit score is not present in the letter.

TEST 2 RESULT:
 {'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


Report your own accuracy numbers from 4.1. Free-text fields like purpose are usually hardest, since exact wording rarely matches the gold text verbatim even when the meaning is correct.
The reliability experiment typically shows temperature 0 gives identical outputs across all 5 runs, while temperature 1.0 gives several distinct outputs, the clear implication is that production extraction pipelines should run at temperature 0, since a system whose "facts" change between identical runs is not trustworthy.
Report your actual PASS/FAIL results from Test 1 and Test 2. If either failed, the fix is prompt-level (stronger "say you don't know" / "return null" instructions) combined with system-level safeguards (validating output against the fixed schema, rejecting unexpected keys or values).

Part 4.4 - Appropriateness: should this system exist?

Applicants who write poorly in English but run genuinely solid businesses could be unfairly harmed: the summarizer/extractor may under-represent their venture due to awkward phrasing, making a good business look weaker than it is.

Loan letters contain names and financial details; sending them to a third-party API hosted abroad raises data-protection and cross-border-transfer concerns (e.g. under Ghana's Data Protection Act) and requires applicant consent. Before deploying: check the provider's data-retention/training-use policy, get compliance sign-off, and confirm the provider does not train on your data.

Two safeguards: (a) mandatory human review of every brief before any outcome is communicated, never fully automated; (b) full audit logging of every prompt/response plus a formal appeal process so an applicant can contest a decision influenced by the system.

Section 5 — Reflection

1. Prompting as engineering. Both are iterative and empirical: change one variable, evaluate, refine. But prompts are natural language, ambiguous, hard to test exhaustively, sensitive to small wording changes, with no gradient to follow, unlike the numeric hyperparameter search in Lab 3.

2. Trust. Not for fully unattended use, by design the system never outputs approve/reject. The result with the biggest influence is usually the hallucination probing outcome: any invented fact under adversarial input is disqualifying regardless of otherwise-good accuracy.

3. Cost and scale. Take your own response.usage.total_tokens for one letter's full pipeline and multiply by 1,000. For example 1,500 tokens/letter * 1,000 = 1.5M tokens/month, comfortably inside Groq's free/low-cost tier.

4. API vs. training your own. An API wins when the task is language-understanding-heavy and a pretrained model already handles it well, six letters is nowhere near enough data to train anything from scratch. Training your own would make sense at very large, narrow, repeated-prediction scale, or when data must never leave your infrastructure.